In [56]:
import chipwhisperer as cw
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error 
from scipy import stats
import sys
import scipy.signal
sys.path.append("/home/40265864@ecit.qub.ac.uk/Toeplitz_attack/")
from privacy_amplification.FFT.FFT import *

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_SAM4S'
SS_VER = 'SS_VER_2_1'

try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

try:
    if SS_VER == "SS_VER_2_1":
        target_type = cw.targets.SimpleSerial2
    elif SS_VER == "SS_VER_2_0":
        raise OSError("SS_VER_2_0 is deprecated. Use SS_VER_2_1")
    else:
        target_type = cw.targets.SimpleSerial
except:
    SS_VER="SS_VER_1_1"
    target_type = cw.targets.SimpleSerial

try:
    target = cw.target(scope, target_type)
except:
    print("INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.")
    print("INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.")
    scope = cw.scope()
    target = cw.target(scope, target_type)


print("INFO: Found ChipWhisperer😍")

if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
    prog = cw.programmers.XMEGAProgrammer
elif "neorv32" in PLATFORM.lower():
    prog = cw.programmers.NEORV32Programmer
elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
    prog = cw.programmers.SAM4SProgrammer
else:
    prog = None

INFO: Found ChipWhisperer😍


In [57]:
import time
time.sleep(0.05)
scope.default_setup()

def reset_target(scope):
    if PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
        scope.io.pdic = 'low'
        time.sleep(0.1)
        scope.io.pdic = 'high_z' #XMEGA doesn't like pdic driven high
        time.sleep(0.1) #xmega needs more startup time
    elif "neorv32" in PLATFORM.lower():
        raise IOError("Default iCE40 neorv32 build does not have external reset - reprogram device to reset")
    elif PLATFORM == "CW308_SAM4S" or PLATFORM == "CWHUSKY":
        scope.io.nrst = 'low'
        time.sleep(0.25)
        scope.io.nrst = 'high_z'
        time.sleep(0.25)
    else:  
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.io.nrst = 'high_z'
        time.sleep(0.05)

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7370129.87012987         
scope.clock.adc_freq                     changed from 0                         to 29480519.48051948        
scope.clock.extclk_monitor_enabled       changed from True                      to False                    
scope.clock.extclk_tolerance             changed from 1144409.1796875           to 13096723.705530167       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2      

In [58]:
scope.adc.samples = 2900                 # Number of samples per segment
scope.adc.stream_mode = "segmented"            # Enable segmented capture
scope.adc.segments = 12          # Total number of segments to capture
scope.adc.offset = 10

# Gain settings
#scope.gain.db = 30

# Trigger settings
#scope.trigger.triggers = "tio4"          # Make sure this matches your trigger pin
#scope.clock.clkgen_freq = 7370000        # Or whatever frequency your target uses
#scope.clock.adc_src = "clkgen_x4"

In [59]:
%%bash -s "$PLATFORM" "$SS_VER"
cd firmware/Toeplitz_FFT
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Welcome to another exciting ChipWhisperer target build!!
Size after:
+--------------------------------------------------------
+ Built for platform Microchip SAM4S with:
+ CRYPTO_TARGET = NONE
+ CRYPTO_OPTIONS = AES128C
   text	   data	    bss	    dec	    hex	filename
  23828	    108	   4140	  28076	   6dac	Toeplitz_FFT-CW308_SAM4S.elf
+--------------------------------------------------------


In [60]:
cw.program_target(scope, prog, "firmware/Toeplitz_FFT/Toeplitz_FFT-{}.hex".format(PLATFORM))

In [6]:
target.write("00010101110101011011\n")

In [7]:
target.write("1010110101011\n")

In [8]:
target.write("01101101111000011100\n")


In [61]:
def align_traces(traces):
    ref_trace = traces[0]  # Use the first trace as reference
    aligned_traces = []

    for trace in traces:
        correlation = np.correlate(trace, ref_trace, mode="full")  # Compute cross-correlation
        shift = np.argmax(correlation) - (len(trace) - 1)  # Find best alignment
        aligned_trace = np.roll(trace, -shift)  # Shift the trace
        aligned_traces.append(aligned_trace)
    
    return np.array(aligned_traces)

In [62]:
def get_trace(x):
    #num_char = target.in_waiting()
    #while num_char > 0:
        #target.read(num_char, 10)
        #time.sleep(0.01)
        #num_char = target.in_waiting()
    time.sleep(0.05)
    #target.flush()
    scope.arm()
    target.write(x)
    if scope.capture():
        raise RuntimeError("Capture failed")
    trace_segments = scope.get_last_trace_segmented()
    alligned_traces = align_traces(trace_segments)
    return alligned_traces


In [13]:
traces = []
reset_target(scope)

##inital toeplitz setup
target.write("10010\n")
time.sleep(0.01)
target.write("1010\n")


trace = get_trace("00000\n")
k=7

In [14]:
fig = cw.plot()
for seg in trace:
    fig *= cw.plot(seg)
fig

:Overlay
   .Curve.I    :Curve   [x]   (y)
   .Curve.II   :Curve   [x]   (y)
   .Curve.III  :Curve   [x]   (y)
   .Curve.IV   :Curve   [x]   (y)
   .Curve.V    :Curve   [x]   (y)
   .Curve.VI   :Curve   [x]   (y)
   .Curve.VII  :Curve   [x]   (y)
   .Curve.VIII :Curve   [x]   (y)
   .Curve.IX   :Curve   [x]   (y)
   .Curve.X    :Curve   [x]   (y)
   .Curve.XI   :Curve   [x]   (y)
   .Curve.XII  :Curve   [x]   (y)
   .Curve.XIII :Curve   [x]   (y)

In [21]:
traces = []
reset_target(scope)

##inital toeplitz setup
##target.write("00010\n")
#time.sleep(0.01)
##target.write("1010\n")


x1 = "00000000\n" # 00 on butterfly 1
x2 = "10000000\n" # 10 on butterfly 1
x3 = "00001000\n" # 01 on butterfly 1
x4 = "10001000\n" # 11 on butterfly 1



for i in range(4):
    warmup = get_trace(x2)

    

for i in range(100):
    traces.append(get_trace(x1))
for i in range(100):
    traces.append(get_trace(x2))
for i in range(100):
    traces.append(get_trace(x3))
for i in range(100):
    traces.append(get_trace(x4))
""""
for i in range(10):
    traces.append(get_trace(x3))
for i in range(10):
    traces.append(get_trace(x4))
for i in range(10):
    traces.append(get_trace(x5))
for i in range(10):
    traces.append(get_trace(x6))
"""

'"\nfor i in range(10):\n    traces.append(get_trace(x3))\nfor i in range(10):\n    traces.append(get_trace(x4))\nfor i in range(10):\n    traces.append(get_trace(x5))\nfor i in range(10):\n    traces.append(get_trace(x6))\n'

In [22]:
traces_x1 = []
traces_x2 = []
traces_x3 = []
traces_x4 = []
for i in range(100):
    traces_x1.append(traces[i])
    traces_x2.append(traces[i+100])
    traces_x3.append(traces[i+200])
    traces_x4.append(traces[i+300])
traces_x1_avg = np.mean(traces_x1, axis=0)
traces_x2_avg = np.mean(traces_x2, axis=0)
traces_x3_avg = np.mean(traces_x3, axis=0)
traces_x4_avg = np.mean(traces_x4, axis=0)

In [23]:
cw.plot(traces_x1_avg[0]) * cw.plot(traces_x2_avg[0]) * cw.plot(traces_x3_avg[0]) * cw.plot(traces_x4_avg[0]) 


:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [ ]:
cw.plot(traces_x2_avg[9] - traces_x1_avg[10])

:Curve   [x]   (y)

In [19]:
cw.plot(traces_x1_avg) * cw.plot(traces_x2_avg)

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [29]:
avg1 = np.mean(traces[0:10],axis=0)
avg2 = np.mean(traces[10:20],axis=0)
avg3 = np.mean(traces[20:30],axis=0)
avg4 = np.mean(traces[30:40],axis=0)
avg5 = np.mean(traces[40:50],axis=0)
avg6 = np.mean(traces[50:60],axis=0)



fig = cw.plot()
fig *= cw.plot(avg6-avg5)
fig

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [17]:
fig = cw.plot()
fig *= cw.plot(traces[45] - traces[55])
fig

NameError: name 'traces' is not defined

In [53]:
def gen_hyp_string_basic(x):
    x_str = list("00000\n")
    x_bin = bin(x)[2:]
    x_bin = x_bin[::-1]
    for i in range(len(x_bin)):
        x_str[int(i)] = x_bin[i]
    x_str = ''.join(x_str)
    return x_str

1. Set a random key to be the correct key & record trace as reference trace
2. For the first 16 bits of the re-ordered key input, make a guess
3. Account for padded bits to reduce key space
4. For each guess, carry out FFT and record trace
5. Compare each guess to reference trace, select max correlation as guess

In [ ]:
##let x be a decimal input for the bits 0,2,4,6,8,10,12,14,16,18
def gen_hyp_string_8_bit_chunk(x, offset):
    x_str = list("00000000000000000000\n")
    x_bin = bin(x)[2:]
    x_bin = x_bin[::-1]
    for i in range(len(x_bin)):
        x_str[int(4*i + offset)] = x_bin[i] ##change 1 to 4 in real scenario
    x_str = ''.join(x_str)
    return x_str

In [57]:
def make_guesses(offset):
    traces = []
    reset_target(scope)
    ##inital toeplitz setup
    target.write("01101\n")
    time.sleep(0.01)
    target.write("1010\n")
    x2 = "11111\n"
    for i in range(4):
        warmup = get_trace(x2)


    for i in range(32):
        if i % 200 == 0 and i != 0:
            target.flush()
            reset_target(scope)
            target.write("01101\n")
            time.sleep(0.01)
            target.write("1010\n")
            x2 = "11111\n"
            for i in range(4):
                warmup = get_trace(x2)

        x = gen_hyp_string_basic(i)
        print("Testing Hyptohesis {0}, input: {1}".format(i,x))
        traces.append(get_trace(x))

    return traces

In [63]:
reset_target(scope)

##inital toeplitz setup
target.write("01101\n")
time.sleep(0.01)
target.write("1010\n")
x2 = "1111\n"
for i in range(4):
    warmup = get_trace(x2)

#x0 = "10001000000010000000\n"
#x0 = "11001101000010010100\n"
#x0 = "10110100100101100101\n" #x0 = "1011 0100 1001 0110 0101\n" ##x0=[10100] x1=[01011] x2=[10010] x3=[10101]
x0 = "01101\n"
time.sleep(0.1)
target_trace = get_trace(x0)


traces0 = make_guesses(0)
#traces2 = make_guesses(2)

Testing Hyptohesis 0, input: 00000

Testing Hyptohesis 1, input: 10000

Testing Hyptohesis 2, input: 01000

Testing Hyptohesis 3, input: 11000

Testing Hyptohesis 4, input: 00100

Testing Hyptohesis 5, input: 10100

Testing Hyptohesis 6, input: 01100

Testing Hyptohesis 7, input: 11100

Testing Hyptohesis 8, input: 00010

Testing Hyptohesis 9, input: 10010

Testing Hyptohesis 10, input: 01010

Testing Hyptohesis 11, input: 11010

Testing Hyptohesis 12, input: 00110

Testing Hyptohesis 13, input: 10110

Testing Hyptohesis 14, input: 01110

Testing Hyptohesis 15, input: 11110

Testing Hyptohesis 16, input: 00001

Testing Hyptohesis 17, input: 10001

Testing Hyptohesis 18, input: 01001

Testing Hyptohesis 19, input: 11001

Testing Hyptohesis 20, input: 00101

Testing Hyptohesis 21, input: 10101

Testing Hyptohesis 22, input: 01101

Testing Hyptohesis 23, input: 11101

Testing Hyptohesis 24, input: 00011

Testing Hyptohesis 25, input: 10011

Testing Hyptohesis 26, input: 01011

Testing Hyp

In [64]:
MSE = []
PC = []
for i in range(32):
    MSE.append(mean_squared_error(np.concatenate(target_trace), np.concatenate(traces0[i])))
    PC.append(stats.pearsonr(np.concatenate(target_trace), np.concatenate(traces0[i])))

    print("Guess {0}. MSE: {1}, PC: {2}".format(i,MSE[i], PC[i]))

Guess 0. MSE: 0.04226197446268427, PC: PearsonRResult(statistic=0.24443525197263738, pvalue=0.0)
Guess 1. MSE: 0.04324114114639029, PC: PearsonRResult(statistic=0.2270227137134085, pvalue=0.0)
Guess 2. MSE: 0.044305333695213465, PC: PearsonRResult(statistic=0.21027646032861913, pvalue=0.0)
Guess 3. MSE: 0.04448111098422413, PC: PearsonRResult(statistic=0.20426067246073673, pvalue=0.0)
Guess 4. MSE: 0.04400011651467201, PC: PearsonRResult(statistic=0.21501247796096618, pvalue=0.0)
Guess 5. MSE: 0.04076617587317596, PC: PearsonRResult(statistic=0.2722043214267587, pvalue=0.0)
Guess 6. MSE: 0.04408385481975655, PC: PearsonRResult(statistic=0.2116042171911953, pvalue=0.0)
Guess 7. MSE: 0.03950610142741533, PC: PearsonRResult(statistic=0.29195700386923507, pvalue=0.0)
Guess 8. MSE: 0.044605168431245366, PC: PearsonRResult(statistic=0.2022288735114199, pvalue=0.0)
Guess 9. MSE: 0.043208479132667656, PC: PearsonRResult(statistic=0.226357316511725, pvalue=0.0)
Guess 10. MSE: 0.0434675008717730

In [69]:
cw.plot(np.concatenate(target_trace) - np.concatenate(traces0[28])) * cw.plot(np.concatenate(target_trace) - np.concatenate(traces0[22]))

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [65]:
print(np.argsort(np.asarray(MSE)))

[22 28 18 30 20 24 26 16 29 25 23 31 14 27  7 15 19 13  5  0 12 21 17  9
  1 10 11  4  6  2  3  8]


In [ ]:
print(gen_hyp_string(27,0))

In [51]:
def array_to_bin_input(arr):
    op = ""
    for x in arr:
        op = op + str(int(x))
    op = op + "\n"
    return op


In [45]:
def infer_from_traces(target_trace, template_traces, segment):
    ##built for 2 bit guesser

    MSE = []
    PC = []
    print("\n\nSegment {0}".format(segment))
    for i in range(4):
        MSE.append(mean_squared_error(target_trace[segment], template_traces[i][segment]))
        PC.append(stats.pearsonr(target_trace[segment],template_traces[i][segment]))
        print("Guess {0}. MSE: {1}, PC: {2}".format(i,MSE[i], PC[i]))
        
    best = np.argsort(MSE)[0]
    result = []
    match best:
        case 0:
            result = [0,0]
        case 1:
            result = [0,1]
        case 2:
            result = [1,0]
        case 3:
            result = [1,1]
    
    return result

In [43]:
def reverse_bits(n, bitSize):
    result = 0
    for i in range(bitSize):
        if n & (1 << i):
            result |= 1 << (bitSize - 1 - i)
    return result

def bit_reverse(x):
    N = x.shape[0]
    s = int(np.log2(N))
    for i in range(N):
        rb = reverse_bits(i,s)
        if(i < rb):
            tmp = x[i]
            x[i] = x[rb]
            x[rb] = tmp
    return x

def gen_pairs(N):
    pairs = []
    order_list = np.arange(N)
    reverse_order = bit_reverse(order_list)
    for i in range(int(N/2)):
        pairs.append([reverse_order[int(2*i)], reverse_order[int(2*i + 1)]])
    return pairs

In [70]:
def guess_sequentially_by_2(target_trace, repeats, N):
    ##modify to generate this on function input N (lenght of FFT)
    pairs = gen_pairs(N)
    key_hyp = np.zeros(N)
    segment = 0

    for pair in pairs:

        reset_target(scope)
        for _ in range(5):
            warmup_str = array_to_bin_input(np.ones(N))
            warmup = get_trace(warmup_str)

        traces_avg = []
        guesses = [[0,0], [0,1], [1,0], [1,1]]
        for guess in guesses:
            key_hyp[pair[0]] = guess[0]
            key_hyp[pair[1]] = guess[1]
            hyp_string = array_to_bin_input(key_hyp)
            traces = []
            for r in range(repeats):
                traces.append(get_trace(hyp_string))
            traces_avg.append(np.mean(traces,axis=0))
        
        result = infer_from_traces(target_trace, traces_avg, segment)
        key_hyp[pair[0]] = result[0]
        key_hyp[pair[1]] = result[1]
        segment += 1
    return key_hyp




In [ ]:
N=32
reset_target(scope)
correct_key = np.random.randint(2,size=N)
correct_key_str = array_to_bin_input(correct_key)
print(correct_key)

for _ in range(5):
    warmup_str = array_to_bin_input(np.ones(N))
    warmup = get_trace(warmup_str)

target_trace = get_trace(correct_key_str)

start = time.time()
guess = guess_sequentially_by_2(target_trace, 20, N)
end = time.time()

time_taken = end - start

guess = np.asarray(guess,dtype=int)
correct_key = np.asarray(correct_key,dtype=int)
print("\n{0} : Eve's Guess".format(guess))
print("{0} : Correct Key".format(correct_key))
if(guess == correct_key).all():
    print("Successfull Key Recovery")
else:
    print("Incorrect Key Recovery")
print("Time Taken: {0}s".format(time_taken))

[0 0 1 0 1 0 0 0 1 0 0 1 1 1 1 0]


Segment 0
Guess 0. MSE: 0.0316138112626227, PC: PearsonRResult(statistic=0.4180091153344868, pvalue=5.460000667250633e-123)
Guess 1. MSE: 0.00013141011078425955, PC: PearsonRResult(statistic=0.9975841056719725, pvalue=0.0)
Guess 2. MSE: 0.035600998321199215, PC: PearsonRResult(statistic=0.3457167603184541, pvalue=3.607816226530807e-82)
Guess 3. MSE: 0.019740402480576444, PC: PearsonRResult(statistic=0.6358871250970058, pvalue=0.0)


Segment 1
Guess 0. MSE: 0.04601220445170942, PC: PearsonRResult(statistic=0.15178689077761004, pvalue=2.0919653037929917e-16)
Guess 1. MSE: 0.019644627771518607, PC: PearsonRResult(statistic=0.6372229826341383, pvalue=0.0)
Guess 2. MSE: 0.04460158037436177, PC: PearsonRResult(statistic=0.17805376457637095, pvalue=4.45923697502021e-22)
Guess 3. MSE: 0.00012528518554711185, PC: PearsonRResult(statistic=0.9976846493052786, pvalue=0.0)


Segment 2
Guess 0. MSE: 0.03101834555187691, PC: PearsonRResult(statistic=0.4276715880612